# Human-in-the-Loop (HITL) 패턴

## 학습 목표
- Human-in-the-Loop 패턴을 `interrupt_before`로 구현할 수 있다
- `MemorySaver` 체크포인터와 함께 그래프를 중단/재개할 수 있다
- `graph.update_state()`로 사람의 결정을 그래프 상태에 반영할 수 있다
- 실전 예제에서 코드 리뷰 에이전트를 구현할 수 있다

## 참조 자료
- 학습 문서: `docs/part5-advanced/11-quality-assurance.md` (섹션 2.5)
- LangGraph HITL: https://langchain-ai.github.io/langgraph/concepts/human_in_the_loop/

## 개념 요약

### interrupt_before 작동 원리

```
[generate] → [review] ──interrupt_before──→ [execute]
                                                ↑
                                    사람이 확인 후
                             graph.invoke(None, config) 로 재개
```

### 핵심 API

| API | 설명 |
|-----|------|
| `graph.compile(checkpointer=MemorySaver(), interrupt_before=["node"])` | 특정 노드 실행 전 중단 설정 |
| `graph.invoke(input, config)` | 그래프 실행 (중단 지점까지) |
| `graph.update_state(config, {"key": value})` | 중단된 상태에 사람의 결정 반영 |
| `graph.invoke(None, config)` | 중단된 지점부터 재개 |

### MemorySaver의 역할

```
MemorySaver = 체크포인터
  → 그래프가 중단될 때 현재 상태를 메모리에 저장
  → thread_id로 상태를 식별
  → 재개 시 저장된 상태에서 이어서 실행
```

In [ ]:
import sys
sys.path.insert(0, '..')

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from common.fake_llm import FakeLLM

print("Human-in-the-Loop 실습 준비 완료")

## 실습 1: interrupt_before 기본 패턴

3개 노드 (generate → review → execute) 로 구성된 그래프를 만들고,
`execute` 노드 실행 전에 사람이 확인할 수 있도록 중단합니다.

**흐름:**
```
[generate] → [review] → [execute]
                 ↑
          interrupt_before=["execute"]
```

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: ApprovalState TypedDict에 task, plan, approved, result 필드 정의
# 힌트 2: generate_node는 FakeLLM으로 plan을 생성 (state["task"]를 입력으로)
# 힌트 3: review_node는 plan을 출력하고 상태를 그대로 반환 (패스스루)
# 힌트 4: execute_node는 plan을 실행하여 result를 반환

class ApprovalState(TypedDict):
    task: str
    plan: str | None
    approved: bool | None
    result: str | None

plan_llm = FakeLLM(responses={
    "default": "1. 데이터베이스 백업\n2. 스키마 변경 적용\n3. 애플리케이션 재시작"
})

def generate_node(state: ApprovalState) -> dict:
    """실행 계획을 생성합니다."""
    # TODO: plan_llm.invoke(state["task"]).content 로 계획 생성
    pass

def review_node(state: ApprovalState) -> dict:
    """생성된 계획을 출력합니다. (interrupt_before 전 마지막 노드)"""
    # TODO: 계획을 출력하고 상태를 그대로 반환 (패스스루)
    pass

def execute_node(state: ApprovalState) -> dict:
    """승인된 계획을 실행합니다."""
    # TODO: result = f"실행 완료: {state['plan']}" 반환
    pass

In [ ]:
# TODO: 그래프 구성 및 컴파일
# 힌트 1: StateGraph(ApprovalState) 생성
# 힌트 2: generate → review → execute 순서로 엣지 연결
# 힌트 3: graph.compile(checkpointer=MemorySaver(), interrupt_before=["execute"])

def build_approval_graph():
    """interrupt_before 기본 패턴 그래프를 구성합니다."""
    graph = StateGraph(ApprovalState)
    
    # TODO: 노드 추가
    
    # TODO: 엣지 연결 (generate → review → execute → END)
    
    # TODO: compile with checkpointer and interrupt_before
    pass

app1 = build_approval_graph()
print("그래프 구성 완료")

In [ ]:
# TODO: 실행 → 중단 → 재개 시나리오
# 힌트 1: config = {"configurable": {"thread_id": "approval-001"}}
# 힌트 2: app1.invoke(초기상태, config) 로 실행 (execute 전에 중단됨)
# 힌트 3: app1.invoke(None, config) 로 재개

config1 = {"configurable": {"thread_id": "approval-001"}}

# Step 1: 그래프 실행 (execute 노드 전에 중단됨)
# TODO: 구현

# Step 2: 사람이 계획을 확인하고 재개 결정
print("\n--- 사람이 계획을 검토합니다 ---")
print("계획이 적절합니다. 실행을 승인합니다.")

# Step 3: 중단된 지점부터 재개
# TODO: app1.invoke(None, config1) 로 재개

In [ ]:
# 검증 셀
# result1은 마지막 invoke 결과
assert result1 is not None, "결과가 None입니다"
assert result1.get("plan") is not None, "plan이 생성되어야 합니다"
assert result1.get("result") is not None, "execute 후 result가 있어야 합니다"
assert "실행 완료" in result1["result"], f"result에 '실행 완료'가 포함되어야 합니다. 현재: {result1['result']}"
print("최종 결과:", result1["result"])
print("✅ 실습 1 완료! interrupt_before로 중단 후 재개가 성공했습니다.")

## 실습 2: 승인/거부 분기

`update_state`로 사람이 승인/거부를 결정하고,
조건부 엣지로 실행 경로를 분기합니다.

**흐름:**
```
[generate_plan] → [approval_gate] ──interrupt_before──→ (check_decision)
                                                              ↓ approved=True
                                                          [execute]
                                                              ↓ approved=False
                                                          [regenerate]
```

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: DecisionState에 task, plan, approved, result, attempt_count 필드 정의
# 힌트 2: approved 필드의 기본값은 None (아직 결정 안 됨)
# 힌트 3: check_decision 함수는 state["approved"] == True 이면 "execute", 아니면 "regenerate"

class DecisionState(TypedDict):
    task: str
    plan: str | None
    approved: bool | None
    result: str | None
    attempt_count: int

def make_plan_node(state: DecisionState) -> dict:
    """계획을 생성합니다. attempt_count를 증가시킵니다."""
    # TODO: 계획 생성 + attempt_count 증가
    pass

def approval_gate_node(state: DecisionState) -> dict:
    """승인 게이트 (패스스루 - interrupt_before로 중단됨)."""
    # TODO: 상태 그대로 반환 (이 노드는 interrupt_before로 중단됨)
    pass

def check_decision(state: DecisionState) -> str:
    """승인 여부에 따라 분기합니다."""
    # TODO: approved == True → "execute", 그 외 → "regenerate"
    pass

def execute_approved_node(state: DecisionState) -> dict:
    """승인된 계획을 실행합니다."""
    # TODO: result = f"[승인됨] 실행 완료: {state['plan']}" 반환
    pass

def regenerate_node(state: DecisionState) -> dict:
    """거부 시 재생성을 알립니다."""
    # TODO: result = f"[거부됨] 재생성이 필요합니다. 시도 {state['attempt_count']}회" 반환
    pass

In [ ]:
# TODO: 그래프 구성
# 힌트 1: make_plan → approval_gate → (조건부) → execute 또는 regenerate
# 힌트 2: interrupt_before=["approval_gate"]
# 힌트 3: add_conditional_edges("approval_gate", check_decision, {"execute": "execute_approved", "regenerate": "regenerate"})

def build_decision_graph():
    """승인/거부 분기 그래프를 구성합니다."""
    graph = StateGraph(DecisionState)
    
    # TODO: 노드 추가
    
    # TODO: 엣지 연결
    
    # TODO: compile
    pass

app2 = build_decision_graph()
print("승인/거부 분기 그래프 구성 완료")

In [ ]:
# 시나리오 A: 승인
# TODO: 여기에 코드를 작성하세요
# 힌트 1: config_approve = {"configurable": {"thread_id": "decision-approve"}}
# 힌트 2: 첫 invoke로 계획 생성 및 중단
# 힌트 3: app2.update_state(config_approve, {"approved": True}) 로 승인 결정 반영
# 힌트 4: app2.invoke(None, config_approve) 로 재개

config_approve = {"configurable": {"thread_id": "decision-approve"}}

# Step 1: 실행
# TODO

# Step 2: 사람이 승인
print("\n--- 사람이 승인합니다 ---")
# TODO: update_state로 approved=True 설정

# Step 3: 재개
# TODO

print("승인 시나리오 결과:", result_approve["result"])

In [ ]:
# 시나리오 B: 거부
# TODO: 여기에 코드를 작성하세요
# 힌트: approved=False로 update_state 후 재개

config_reject = {"configurable": {"thread_id": "decision-reject"}}

# Step 1: 실행
# TODO

# Step 2: 사람이 거부
print("\n--- 사람이 거부합니다 ---")
# TODO: update_state로 approved=False 설정

# Step 3: 재개
# TODO

print("거부 시나리오 결과:", result_reject["result"])

In [ ]:
# 검증 셀
assert result_approve is not None, "승인 결과가 None입니다"
assert result_reject is not None, "거부 결과가 None입니다"
assert "승인" in result_approve["result"], f"승인 결과에 '승인됨'이 포함되어야 합니다. 현재: {result_approve['result']}"
assert "거부" in result_reject["result"], f"거부 결과에 '거부됨'이 포함되어야 합니다. 현재: {result_reject['result']}"
print("승인 결과:", result_approve["result"])
print("거부 결과:", result_reject["result"])
print("✅ 실습 2 완료! update_state로 승인/거부 분기가 올바르게 작동합니다.")

## 실습 3: 실전 예제 - 코드 리뷰 에이전트

코드를 자동 생성하고 사람이 검토한 후 배포 또는 수정을 결정하는 에이전트를 구현합니다.

**흐름:**
```
[generate_code] → [human_review] ──interrupt_before──→ (route_decision)
                                                              ↓ approved=True
                                                          [deploy]
                                                              ↓ approved=False
                                                          [fix_code]
                                                              ↓
                                                          [human_review] (반복)
```

In [ ]:
# TODO: 전체 HITL 코드 리뷰 에이전트를 구현하세요
# 힌트 1: CodeReviewState에 필드 정의
#   - requirement: str (개발 요구사항)
#   - code: str | None (생성된 코드)
#   - review_feedback: str | None (사람의 피드백)
#   - approved: bool | None (승인 여부)
#   - deploy_result: str | None (배포 결과)
#   - revision_count: int (수정 횟수)
# 힌트 2: generate_code_node - FakeLLM으로 코드 생성
# 힌트 3: human_review_node - 패스스루 (interrupt_before로 중단됨)
# 힌트 4: route_decision - approved 여부에 따라 "deploy" 또는 "fix_code"
# 힌트 5: deploy_node - 코드를 배포하고 deploy_result 반환
# 힌트 6: fix_code_node - 피드백을 반영하여 코드 수정 (FakeLLM 활용)
# 힌트 7: interrupt_before=["human_review"] 로 컴파일
# 힌트 8: fix_code → human_review (순환) 으로 반복 가능

class CodeReviewState(TypedDict):
    requirement: str
    code: str | None
    review_feedback: str | None
    approved: bool | None
    deploy_result: str | None
    revision_count: int

code_gen_llm = FakeLLM(responses={
    "수정": "def login(user, password):\n    # 피드백 반영: 입력 검증 추가\n    if not user or not password:\n        raise ValueError('입력값이 없습니다')\n    return authenticate(user, password)",
    "default": "def login(user, password):\n    return authenticate(user, password)"
})

def generate_code_node(state: CodeReviewState) -> dict:
    """요구사항에 따라 코드를 생성합니다."""
    # TODO: code_gen_llm으로 code 생성, revision_count=0으로 초기화
    pass

def human_review_node(state: CodeReviewState) -> dict:
    """인간 리뷰 게이트 (interrupt_before로 중단됨)."""
    # TODO: 현재 코드를 출력하고 상태 그대로 반환
    pass

def route_decision(state: CodeReviewState) -> str:
    """승인 여부에 따라 분기합니다."""
    # TODO: approved=True → "deploy", 그 외 → "fix_code"
    pass

def deploy_node(state: CodeReviewState) -> dict:
    """코드를 배포합니다."""
    # TODO: deploy_result = f"배포 완료 (수정 {revision_count}회)\n{code}" 반환
    pass

def fix_code_node(state: CodeReviewState) -> dict:
    """피드백을 반영하여 코드를 수정합니다."""
    # TODO: code_gen_llm.invoke(f"수정 요청: {review_feedback}").content 로 코드 수정
    # TODO: revision_count 증가, approved=None으로 리셋 (다음 리뷰를 위해)
    pass

In [ ]:
# TODO: 코드 리뷰 에이전트 그래프 구성
# 힌트 1: generate_code → human_review → (route_decision) → deploy 또는 fix_code
# 힌트 2: fix_code → human_review (순환 엣지!)
# 힌트 3: interrupt_before=["human_review"]

def build_code_review_graph():
    """코드 리뷰 HITL 에이전트 그래프를 구성합니다."""
    graph = StateGraph(CodeReviewState)
    
    # TODO: 노드 추가
    
    # TODO: 엣지 연결 (순환 포함)
    
    # TODO: compile
    pass

app3 = build_code_review_graph()
print("코드 리뷰 에이전트 그래프 구성 완료")

In [ ]:
# 시나리오: 거부 → 수정 → 승인
# TODO: 여기에 코드를 작성하세요
# 힌트 1: thread_id="code-review-001"
# 힌트 2: 1차 리뷰 → 거부 (피드백: "입력 검증이 없습니다")
#   - update_state({"approved": False, "review_feedback": "입력 검증이 없습니다"})
# 힌트 3: 코드 수정 후 2차 리뷰 → 승인
#   - update_state({"approved": True, "review_feedback": None})

config3 = {"configurable": {"thread_id": "code-review-001"}}

initial_state = {
    "requirement": "로그인 함수 구현",
    "code": None,
    "review_feedback": None,
    "approved": None,
    "deploy_result": None,
    "revision_count": 0,
}

# Step 1: 코드 생성 (human_review 전 중단)
print("=== Step 1: 코드 생성 ===")
# TODO

# Step 2: 1차 리뷰 - 거부
print("\n=== Step 2: 1차 리뷰 - 거부 ===")
# TODO: update_state로 거부 결정 반영

# Step 3: 코드 수정 후 2차 리뷰 전 중단
print("\n=== Step 3: 코드 수정 후 2차 리뷰 ===")
# TODO

# Step 4: 2차 리뷰 - 승인
print("\n=== Step 4: 2차 리뷰 - 승인 ===")
# TODO: update_state로 승인 결정 반영

# Step 5: 배포
print("\n=== Step 5: 배포 ===")
# TODO

In [ ]:
# 검증 셀
assert result3 is not None, "최종 결과가 None입니다"
assert result3.get("deploy_result") is not None, "배포 결과가 없습니다"
assert "배포 완료" in result3["deploy_result"], f"배포 완료 메시지가 없습니다. 현재: {result3['deploy_result']}"
assert result3.get("revision_count", 0) >= 1, "코드 수정이 최소 1회 이루어져야 합니다"
print("최종 배포 결과:")
print(result3["deploy_result"])
print(f"\n총 수정 횟수: {result3['revision_count']}회")
print("✅ 실습 3 완료! 코드 리뷰 HITL 에이전트가 올바르게 작동합니다.")

## 정리

이번 실습에서 배운 내용:

| 개념 | 설명 | 사용법 |
|------|------|--------|
| `MemorySaver` | 그래프 상태를 메모리에 저장하는 체크포인터 | `compile(checkpointer=MemorySaver())` |
| `interrupt_before` | 특정 노드 실행 전 그래프 중단 | `compile(interrupt_before=["node_name"])` |
| `invoke(None, config)` | 중단된 지점부터 재개 | `graph.invoke(None, config)` |
| `update_state` | 중단된 상태에 사람의 결정 반영 | `graph.update_state(config, {"key": value})` |
| `thread_id` | 여러 실행을 구분하는 식별자 | `{"configurable": {"thread_id": "..."}}`  |

### HITL 패턴의 핵심

```
1. compile(interrupt_before=["gate_node"]) 로 중단 지점 지정
2. invoke(initial_state, config) 로 중단까지 실행
3. update_state(config, {"approved": True/False}) 로 결정 반영
4. invoke(None, config) 로 재개
```

## 다음 단계
- Module 12: 멀티 에이전트 시스템